# JetBot - Data collection without gamecontroller

In this notebook we'll collect training data for CNN VAE. The training data save to dataset directory.

## Import module



In [ ]:
import os
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg
import time
import cv2
import numpy as np

## Show log_button

If you enable log_button then start recording images.


In [ ]:
log_button = widgets.ToggleButton(value=False, description='enable logging')
display(log_button)

## Initialize Camera

Next is initializing camera module. Image size is now 1280 x 960 for higher quality. Frame rate is about 27Hz. We'll save image in camera observer method. camera observer method can get image per frame rate. Thus, frame rate is decide to image save interval.

In [ ]:
camera = Camera.instance(width=1280, height=960)
image = widgets.Image(format='jpeg', width=320, height=240)  # Display smaller for UI
camera_link = traitlets.dlink((camera,'value'), (image,'value'), transform=bgr8_to_jpeg)

## UI Widget


In [ ]:
DATASET_DIR = 'dataset'
try:
    os.makedirs(DATASET_DIR)
except FileExistsError:
    print('Directories not created becasue they already exist')

dataset=DATASET_DIR
layout = widgets.Layout(width='100px', height='64px')
count_box   = widgets.IntText(layout=layout, value=len(os.listdir(dataset)))
count_label = widgets.Label(layout=layout, value='Number image:')
count_panel = widgets.HBox([count_label,count_box])

panel = widgets.VBox([count_panel])
display(widgets.HBox([panel,image]))

## Set callback for collect the training data.

```save_record``` is callback for training data. The method set to camera observer. This callback saving the image to DATASET_DIR. When click ```enable logging``` button, this method recording training data. You can check number of training data with ```Number image text box```.

Added: Frame skipping (every 5th frame), fisheye undistortion (replace with your calibrated K and D), and basic augmentation (brightness/contrast).

In [ ]:
import os
from uuid import uuid1

# Calibration matrices for IMX219 fisheye (replace with your actual calibration values)
K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])  # Example: fx=500, fy=500, cx=640, cy=480
D = np.array([k1, k2, p1, p2])  # Example distortion coeffs

frame_counter = 0
skip_frames = 5  # Save every 5th frame to reduce redundancy

def save_record(change):                
    global frame_counter
    if log_button.value:
        frame_counter += 1
        if frame_counter % skip_frames != 0:
            return
        
        image = change['new']
        
        # Undistort fisheye
        undistorted = cv2.fisheye.undistortImage(image, K, D, None, K)
        
        # Simple augmentation: random brightness/contrast
        alpha = 1.0 + np.random.uniform(-0.3, 0.3)  # Contrast
        beta = np.random.uniform(-50, 50)  # Brightness
        augmented = cv2.convertScaleAbs(undistorted, alpha=alpha, beta=beta)
        
        image_name = '{}.jpg'.format(uuid1())
        image_path = os.path.join(DATASET_DIR, image_name)
        save_image = bgr8_to_jpeg(augmented)
        with open(image_path, 'wb') as f:
            f.write(save_image)
        count_box.value = len(os.listdir(dataset)) 


save_record({'new': camera.value})
camera.observe(save_record, names='value')

## Cleanup

After collecting enough data. cleanup camera observer and stop all motor.

In [ ]:
camera.unobserve(save_record, names='value')
camera_link.unlink()
camera.stop()

## Cleate dataset.zip file 

In [ ]:
import datetime
def timestr():
    return str(datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))

!zip -r -q jetbot_{DATASET_DIR}_{timestr()}.zip {DATASET_DIR}